# Context（运行上下文）
Context用于在运行时传递配置参数、用户信息等会话临时数据

## 定义Context Schema

In [ ]:
# 方式一：使用dataclass定义context
from dataclasses import dataclass
@dataclass
class UserContext:
    """Agent运行时上下文"""
    user_id: str = ""

In [3]:
# 方式二：继承TypedDict定义context
from typing_extensions import TypedDict
class UserContext2(TypedDict):
    """运行时上下文类型"""
    user_id: str = ""

## 在Tool中使用Context
在tool中利用runtime来访问Context

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

@tool
def get_users(runtime: ToolRuntime[UserContext]):
    """查询所有用户信息"""
    # 获取store
    store = runtime.store
    if store is None:
        return "Store not available"
    # 获取当前用户信息
    user_id = runtime.context.user_id
    if user_id is None:
        return "当前用户未登录"

    # 获取store中的user_id
    user = store.get(("users", ), user_id)
    if user is None:
        return "当前用户未登录"

    # 校验权限，至少是3级权限
    user_info = dict(user.value)
    if user_info['clearance_level'] < 3:
        return "权限不足"

    # 查询用户
    results = runtime.store.search(("users", ))
    if results is None or len(results) == 0:
        return "未查询到用户"

    users = [item.value for item in results]
    return users

@tool
def get_user_preferences(runtime: ToolRuntime[UserContext]):
    """查询当前用户的偏好，根据偏好输出结果"""
    # 获取当前用户id
    user_id = runtime.context.user_id

    # 获取用户偏好
    user_preference = runtime.store.get(("preferences", ), user_id)

    if user_preference is None:
        return "未查找到用户偏好信息"

    return user_preference.value

## 给Agent添加Context

In [ ]:
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore

memory_store = InMemoryStore()

# 向store中存储数据
memory_store.put(
    ("preferences", ),   # namespace 是一个tuple
    "user_001",  # key 可以是任意类型
    {
        # value, 是JSON格式文档
        "style": "business_markdown",
        "language": "zh-CN"
    }
)

memory_store.put(
    ("preferences", ), 
    "user_002", 
    {
        "style": "trump",
        "language": "en-US"
    }
)

agent = create_agent(
    model="deepseek-chat",
    tools=[get_users, get_user_preferences],
    store = memory_store,
    context_schema=UserContext,
    system_prompt="""
    # indentify
    你是一个热心的助手，你可以调用工具获取用户信息，用户偏好。
    # instruction
    请务必按照用户偏好风格展示结果。
    """
)

调用Agent时，可以传递Context信息

In [ ]:
from langchain.messages import HumanMessage
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询所有用户信息")]},
    context=UserContext(user_id="user_001")
)

for message in response["messages"]:
    message.pretty_print()